# Часть 3 — Langfuse: смотрим, что агент делает внутри

Агент — это цепочка вызовов LLM и инструментов, которую мы не пишем руками: её строит модель.
Чтобы понимать, **что** он сделал, **почему** ответил так, **сколько** это стоило и **где** тормозит,
нужна наблюдаемость (observability). Стандартный инструмент для LLM-приложений — **Langfuse**
(open source, можно поднять у себя — мы так и сделали в `docker-compose.yml`).

| понятие | что это |
|---|---|
| **трейс (trace)** | одна операция целиком: «вопрос → ответ» |
| **наблюдение (observation)** | шаг внутри трейса; шаги вкладываются друг в друга, получается дерево |
| **генерация (generation)** | наблюдение-вызов LLM: промпт, ответ, модель, токены, время, стоимость |
| **сессия (session)** | несколько трейсов одного разговора |
| **оценка (score)** | оценка ответа: от пользователя, по правилам или другой LLM |

Блокнот из двух половин:

- **А. Азбука** — Langfuse на игрушечных примерах: как устроен каждый «кирпичик» и все его аргументы;
- **Б. Трейсим агента** — те же кирпичики на нашем риск-менеджере.

Откройте **http://localhost:3000** — логин `admin@example.com`, пароль `lesson3-admin`, раздел **Tracing**.
Для половины А нужен только Langfuse (`docker compose up -d`) и ключ LLM.

# А. Азбука Langfuse

## А0. Подключение

`get_client()` создаёт клиента. Адрес и ключи он берёт из переменных окружения (у нас — из `.env`):

| переменная | что это |
|---|---|
| `LANGFUSE_HOST` | адрес сервера Langfuse (`http://localhost:3000` или облако) |
| `LANGFUSE_PUBLIC_KEY` | публичный ключ проекта (`pk-lf-…`) — в какой проект писать |
| `LANGFUSE_SECRET_KEY` | секретный ключ (`sk-lf-…`) — пароль к проекту |

`auth_check()` — пробный запрос: сервер отвечает, ключи подходят.

In [1]:
from dotenv import load_dotenv
from langfuse import get_client

load_dotenv()                     # LANGFUSE_HOST / PUBLIC_KEY / SECRET_KEY from .env
langfuse = get_client()
print("✅ Langfuse доступен" if langfuse.auth_check() else "❌ Langfuse не отвечает: docker compose up -d")

✅ Langfuse доступен


## А1. Один шаг

Главный кирпичик — `start_as_current_observation`: «начать шаг и сделать его текущим». Пока работает
блок `with`, шаг открыт; на выходе — закрыт, Langfuse запоминает время.
Шаг без родителя автоматически создаёт новый **трейс**.

| аргумент | что это |
|---|---|
| `name` | имя шага — так он будет подписан в дереве |
| `as_type` | тип: `"span"` (по умолчанию, просто шаг), `"generation"` (вызов LLM), `"agent"`, `"tool"`, `"chain"`… Влияет на иконку и фильтры |
| `input` | что пришло на вход: строка, число, словарь, список сообщений — любой JSON |
| `output` | результат (обычно его ставят позже через `.update`) |
| `metadata` | любые доп. поля для поиска и фильтров: `{"region": "EU"}` |
| `level` | `"DEBUG"` / `"DEFAULT"` / `"WARNING"` / `"ERROR"` — важность шага |
| `status_message` | пояснение, обычно текст ошибки |

`span.update(...)` — дописать к открытому шагу то, что стало известно позже (тот же набор аргументов).

In [2]:
with langfuse.start_as_current_observation(name="привет", input="2 + 2") as span:
    result = 2 + 2
    span.update(output=result)                    # the result is known only now
    trace_id = langfuse.get_current_trace_id()    # the id of the trace we are in

langfuse.flush()                                  # send now (normally it is sent in the background)
print("🔎", langfuse.get_trace_url(trace_id=trace_id))

🔎 http://localhost:3000/project/lesson3/traces/433424811c1e8b2910052562b798276e


- `get_current_trace_id()` — номер текущего трейса (внутри блока `with`);
- `get_trace_url(trace_id=...)` — ссылка на трейс в интерфейсе;
- `flush()` — отправить накопленное сразу. Обычно данные уходят в фоне пачками; в блокноте мы хотим
  увидеть трейс немедленно.

Откройте ссылку: один шаг «привет», вход `2 + 2`, выход `4`, время.

## А2. Вложенные шаги

Шаг, открытый **внутри** другого, становится его ребёнком — так строится дерево.

In [3]:
import time

# from trace_view import print_trace                # prints a trace as a tree (same as the UI)

with langfuse.start_as_current_observation(name="оформить заказ", input={"order": 42}) as order:
    with langfuse.start_as_current_observation(name="проверить склад", input={"item": "кофе"}) as step:
        time.sleep(0.3)                           # pretend to work
        step.update(output="есть 5 шт.")
    with langfuse.start_as_current_observation(name="списать деньги", input={"sum": 300}) as step:
        time.sleep(0.2)
        step.update(output="ok")
    order.update(output="заказ оформлен")
    trace_id = langfuse.get_current_trace_id()

# print_trace(trace_id)

Видно, что внутри заказа и сколько занял каждый шаг. Ровно так потом будет выглядеть агент:
«агент → вызов LLM → инструмент → вызов LLM…».

## А3. Генерация — вызов LLM

Для вызова модели — тип `"generation"`. У него есть свои поля, по которым Langfuse считает
токены и стоимость:

| аргумент | что это |
|---|---|
| `model` | имя модели — по нему Langfuse находит цену |
| `model_parameters` | параметры вызова: `{"temperature": 0, "max_tokens": 100}` |
| `input` | промпт — обычно список сообщений `[{"role": "user", "content": "..."}]` |
| `output` | ответ модели |
| `usage_details` | токены: `{"input": 12, "output": 3}` |
| `cost_details` | стоимость, если хотите указать её сами: `{"input": 0.0001, "output": 0.0002}` |

Здесь мы заполняем всё **вручную** — чтобы увидеть, из чего состоит генерация.

In [ ]:
import os

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=os.getenv("MODEL", "openai/gpt-4o-mini"), base_url="https://openrouter.ai/api/v1",
                 api_key=os.getenv("OPENROUTER_API_KEY"))
question = "Столица Чехии? Одно слово."

with langfuse.start_as_current_observation(
        name="вопрос модели", as_type="generation",
        model=os.getenv("MODEL"), model_parameters={"temperature": 0},
        input=[{"role": "user", "content": question}]) as generation:
    reply = llm.invoke(question)
    generation.update(output=reply.text,
                      usage_details={"input": reply.usage_metadata["input_tokens"],     # tokens from the API
                                     "output": reply.usage_metadata["output_tokens"]})
    trace_id = langfuse.get_current_trace_id()

print("ответ:", reply.text)
# print_trace(trace_id)

ответ: Прага


NameError: name 'print_trace' is not defined

## А4. Свойства трейса

Шаги описывают «что происходило», а свойства — «чей это трейс». Их задаёт `propagate_attributes`:
всё, что внутри блока, получает эти свойства.

| аргумент | что это | зачем |
|---|---|---|
| `trace_name` | имя трейса | по нему ищут в списке **Tracing** |
| `session_id` | номер сессии (разговора) | раздел **Sessions**: все трейсы разговора лентой |
| `user_id` | кто пользователь | раздел **Users**: история и расходы по пользователю |
| `tags` | метки | фильтры: `["prod", "lesson3"]` |
| `metadata` | любые поля | фильтры и поиск |
| `version` | версия вашего агента/промпта | сравнивать качество версий |

In [5]:
from langfuse import propagate_attributes

with langfuse.start_as_current_observation(name="шаг", input="hi") as span, \
        propagate_attributes(trace_name="демо-свойства", session_id="demo-session", user_id="student",
                             tags=["lesson3", "demo"], metadata={"course": "ai-agents"}, version="1.0"):
    span.update(output="ok")
    trace_id = langfuse.get_current_trace_id()

print("🔎", langfuse.get_trace_url(trace_id=trace_id))

🔎 http://localhost:3000/project/lesson3/traces/301e6ad13d386cf39755abf2dfd6109b


## А5. `@observe` — трейсинг обычных функций

Чтобы не писать `with` в каждой функции, есть декоратор. Вход (аргументы) и выход (результат)
он записывает сам, вложенность — тоже: функция, вызванная из другой, становится её ребёнком.

| аргумент | что это |
|---|---|
| `name` | имя шага (по умолчанию — имя функции) |
| `as_type` | тип шага, как в А1 |
| `capture_input` / `capture_output` | записывать ли аргументы / результат (`False` — для секретов и больших данных) |

Работает и с `async def`.

In [ ]:
from langfuse import observe


@observe()                                    # name = "add"
def add(a: int, b: int) -> int:
    return a + b


@observe(name="двойной расчёт")
def calc(x: int) -> int:
    return add(x, x) * 10                     # add() becomes a child step


with langfuse.start_as_current_observation(name="демо-observe") as span:
    print("результат:", calc(4))
    trace_id = langfuse.get_current_trace_id()

# print_trace(trace_id)

результат: 80


NameError: name 'print_trace' is not defined

## А6. `CallbackHandler` — LangChain и LangGraph сами

В А3 мы заполняли генерацию руками. Для LangChain/LangGraph это не нужно: передайте в `config`
обработчик `CallbackHandler()` — и он сам запишет каждый вызов модели (промпт, ответ, модель, токены),
каждый инструмент и каждый узел графа.

In [ ]:
from langfuse.langchain import CallbackHandler

with langfuse.start_as_current_observation(name="LangChain + CallbackHandler") as span:
    reply = llm.invoke("Столица Словакии? Одно слово.", config={"callbacks": [CallbackHandler()]})
    span.update(output=reply.text)
    trace_id = langfuse.get_current_trace_id()

# print_trace(trace_id)

NameError: name 'print_trace' is not defined

Одна строка — и генерация со всеми полями из А3 появилась сама. Именно так мы будем трейсить агента.

## А7. Оценки

Оценка (score) — число, категория или да/нет, прикреплённое к трейсу или к одному шагу.

| аргумент | что это |
|---|---|
| `trace_id` | к какому трейсу |
| `observation_id` | к какому шагу (необязательно — тогда ко всему трейсу) |
| `session_id` | или ко всей сессии |
| `name` | название оценки: `"correct"`, `"user-feedback"`, `"tone"` |
| `value` | значение: `1`, `0.8` или строка для категорий |
| `data_type` | `"NUMERIC"` (число), `"CATEGORICAL"` (категория), `"BOOLEAN"` (да/нет) |
| `comment` | пояснение |

In [8]:
answer_trace = trace_id                                       # the trace from А6
langfuse.create_score(trace_id=answer_trace, name="correct", value=1, data_type="NUMERIC",
                      comment="Братислава — верно")
langfuse.create_score(trace_id=answer_trace, name="tone", value="neutral", data_type="CATEGORICAL")
langfuse.flush()
print("🔎", langfuse.get_trace_url(trace_id=answer_trace), "— вкладка Scores (появится через пару секунд)")

🔎 http://localhost:3000/project/lesson3/traces/bfb3c2f10e660a6aa6863865c62b58a9 — вкладка Scores (появится через пару секунд)


## Конструктор: что когда использовать

| задача | кирпичик |
|---|---|
| отметить шаг своего кода | `with langfuse.start_as_current_observation(name=..., input=...) as span:` → `span.update(output=...)` |
| то же для целой функции | `@observe()` |
| вызов LLM без LangChain | `as_type="generation"` + `model`, `usage_details` |
| LangChain / LangGraph | `config={"callbacks": [CallbackHandler()]}` |
| назвать трейс, сессия, пользователь, теги | `with propagate_attributes(...)` |
| оценить ответ | `langfuse.create_score(...)` |
| получить номер и ссылку | `get_current_trace_id()`, `get_trace_url(trace_id=...)` |
| отправить прямо сейчас | `langfuse.flush()` |

---

# Б. Трейсим агента

Теперь те же кирпичики — на риск-менеджере из части 2. В разделе **Tracing** уже есть трейсы
`analyst`: аналитик пишет их с самого начала (метод `answer` в `analyst_agent.py`).

**Перед запуском** должны работать (каждое — в своём терминале, из папки занятия):

```bash
docker compose up -d                  # база банка + Langfuse (один раз)
uv run python mcp_server.py           # терминал 1: MCP-сервер с инструментами → :8101
uv run python analyst_agent.py        # терминал 2: агент-аналитик по A2A     → :8102
```

In [9]:
import httpx

for name, url in [("MCP-сервер", "http://localhost:8101/mcp"), ("аналитик", "http://localhost:8102/.well-known/agent-card.json")]:
    try:
        httpx.get(url, timeout=3)
        print(f"✅ {name} работает")
    except httpx.ConnectError:
        print(f"❌ {name} не запущен — см. команды выше")

✅ MCP-сервер работает
✅ аналитик работает


## Шаг 1. Риск-менеджер из части 2

In [10]:
import os

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()
llm = ChatOpenAI(model=os.getenv("MODEL", "openai/gpt-4o-mini"), base_url="https://openrouter.ai/api/v1",
                 api_key=os.getenv("OPENROUTER_API_KEY"))

from risk_agent import build_risk_agent        # the same agent as in 02_two_agents, kept in a file

risk_manager = build_risk_agent(llm)

## Шаг 2. Первый трейс

Нужны две вещи:

1. **корневой span** `start_as_current_observation` — «папка» трейса: имя, вход, выход;
2. **`CallbackHandler`** в конфиге графа — LangChain/LangGraph сам отчитывается в Langfuse о каждом
   шаге: узлы графа, вызовы LLM, инструменты.

In [11]:
from langfuse import propagate_attributes
from langfuse.langchain import CallbackHandler


async def decide(question: str, chat: str) -> str:
    """Run the risk manager inside a Langfuse trace; return the trace id.

    `chat` is the conversation: the risk manager's memory (thread_id) and the Langfuse session.
    """
    with langfuse.start_as_current_observation(name="risk-manager", as_type="agent", input=question) as root, \
            propagate_attributes(trace_name="risk-manager", session_id=chat, user_id="student"):
        result = await risk_manager.ainvoke({"messages": [("user", question)]},
                                            {"configurable": {"thread_id": chat},
                                             "callbacks": [CallbackHandler()]})      # ← report every step
        answer = result["messages"][-1].text
        root.update(output=answer)
        trace_id = langfuse.get_current_trace_id()
    print("\n⚖️", answer)
    print("\n🔎 трейс:", langfuse.get_trace_url(trace_id=trace_id))
    return trace_id


trace_id = await decide("Можно ли выдать счёту 2378 кредит 150 000 CZK на 24 месяца?", chat="client-2378")

   📨 → аналитику: Для счёта 2378 укажи общий приход за период 01.01.1998–31.12.1998 и средний месячный приход за этот период (CZK).
   📨 → аналитику: Для счёта 2378 укажи средний баланс за период 01.01.1998–31.12.1998 (CZK).
   📨 → аналитику: Для счёта 2378 перечисли все кредиты за всё время и их статусы; отдельно укажи, есть ли кредиты в статусе running_in_debt или finished_default.
   📨 ← аналитик: Счёт 2378, 01.01.1998–31.12.1998:

- Общий приход: **366 959 CZK**
- Средний месячный приход: **30 579,92 CZK** (общий приход ÷ 12)
   📨 ← аналитик: По счёту **2378** кредитов не найдено. Статусы **running_in_debt** и **finished_default** отсутствуют.
   📨 ← аналитик: Средний баланс счёта 2378 за 1998 год: **52 962,26 CZK**.

⚖️ **ОДОБРИТЬ.**

1. **Платёж относительно дохода:** платёж по кредиту — **6 784,09 CZK/мес.** Это меньше 30% среднего месячного прихода за 1998 год (**30 579,92 × 30% = 9 173,98 CZK**). Условие выполнено.
2. **Кредитная история:** кредитов со статусами `running_in_de

Откройте ссылку 🔎. Что там видно:

- **дерево шагов**: `risk-manager` → `LangGraph` → `model` (вызов LLM) → `tools` → `ask_analyst`…;
- у каждой **генерации** (`ChatOpenAI`): полный промпт с системным сообщением, ответ модели,
  `tool_calls`, модель, **токены**, **время**;
- **стоимость** Langfuse считает сам для известных моделей (OpenAI, Anthropic, …). Для модели,
  которой нет в его справочнике, будет 0 — цену можно добавить в *Settings → Models*.

То же дерево можно вывести прямо здесь — функция из `trace_view.py` берёт трейс через API Langfuse:

In [ ]:
from trace_view import print_trace

print_trace(trace_id)

## Шаг 3. Сессии

Трейсы одного разговора объединяются по `session_id` (мы передали его в `propagate_attributes`).
Задаём уточняющий вопрос в том же чате — риск-менеджер помнит клиента (его память, часть 2),
а Langfuse положит трейс в ту же сессию:

In [12]:
await decide("А если 300 000 на 12 месяцев?", chat="client-2378")
print("\n🗂 сессия:", f"{os.getenv('LANGFUSE_HOST')}/project/lesson3/sessions/client-2378")


⚖️ **ОТКАЗАТЬ.**

1. **Платёж относительно дохода:** платёж — **26 096,53 CZK/мес.** Максимум по политике — 30% среднего месячного прихода: **9 173,98 CZK**. Условие не выполнено.
2. **Кредитная история:** кредитов со статусами `running_in_debt` или `finished_default` за всё время не найдено. Условие выполнено.
3. **Средний баланс:** **52 962,26 CZK**, что больше ежемесячного платежа. Условие выполнено.

🔎 трейс: http://localhost:3000/project/lesson3/traces/44571bee4fede9bc5ffd7e956c797118

🗂 сессия: http://localhost:3000/project/lesson3/sessions/client-2378


В разделе **Sessions** — весь разговор по клиенту одной лентой: оба вопроса риск-менеджеру
и все ответы аналитика (номер сессии пришёл ему в `metadata` A2A-сообщения). Так разбирают целые
диалоги пользователей с агентами.

## Шаг 4. Оценки

Langfuse хранит не только «что произошло», но и «хорошо ли получилось». Оценку можно поставить
из кода — например, по кнопке 👍/👎 в интерфейсе:

In [13]:
langfuse.create_score(trace_id=trace_id, name="user-feedback", value=1, comment="👍 решение верное")
langfuse.flush()
print("оценка сохранена:", langfuse.get_trace_url(trace_id=trace_id))

оценка сохранена: http://localhost:3000/project/lesson3/traces/36c01565c554d428dade4a6e8dd6620d


Оценки видны в трейсе и в разделе **Scores**: по ним считают качество агента во времени, ищут плохие
ответы и собирают примеры для улучшения промпта.

## А где работа аналитика?

Посмотрите на дерево из шага 2: `ask_analyst` занял несколько секунд, но **что внутри** — не видно.
Аналитик свои шаги записал, но в **отдельные трейсы**:

In [14]:
for trace in langfuse.api.trace.list(limit=6).data:
    print(f"{trace.timestamp:%H:%M:%S}  {trace.name or '-':<14} {trace.latency or 0:5.1f} s   сессия: {trace.session_id}")

18:04:03  risk-manager     6.1 s   сессия: client-2378
17:59:04  analyst          5.9 s   сессия: client-2378
17:59:04  analyst          5.8 s   сессия: client-2378
17:59:04  analyst          6.5 s   сессия: client-2378
17:59:01  risk-manager    13.7 s   сессия: client-2378
17:55:19  LangChain + CallbackHandler   1.9 s   сессия: None


Рядом с одним `risk-manager` — несколько `analyst`. Сессия у них общая, так что разговор целиком
найти можно. Но **какой** ответ аналитика к **какому** вызову `ask_analyst` относится и что аналитик
делал внутри — из этого списка не понять: это разные процессы, и связи «шаг внутри шага» между ними
нет. С десятком агентов разобраться в таком будет невозможно.

## Итоги

- трейс = корневой span + `CallbackHandler` в конфиге графа — и виден каждый шаг агента;
- генерации: промпт, ответ, токены, время, стоимость;
- сессии собирают разговор, оценки — качество.

Дальше — часть 4: склеиваем трейсы двух агентов в один.